# 02-01 ETL Summary

Reads all minute bars from `minute_data_final/` with **Spark**, derives the trading day from the epoch-ms timestamp, ranks rows per (symbol, day), and writes two summary tables to S3:

- `summary/daily_volume/data.parquet` -- per-symbol daily volume/trades/close
- `summary/minute_summary/data.parquet` -- first + last minute bar per day

These summaries feed every downstream strategy stage (03-01, 04-01, 05-*).

In [ ]:
# ============================================================================
# SETUP -- installs, imports, config (env vars / config.json -- never hardcoded)
# ============================================================================

# --- Install packages (no-op if already present) --------------------------
# !pip install -q duckdb --upgrade
# PySpark is preinstalled on Kaggle
import os
import json
import sys
import time
from io import BytesIO
from pathlib import Path

import numpy as np
import pandas as pd
import boto3
import duckdb
# --- Configuration --------------------------------------------------------
# Secrets resolve in priority order:
#   1. Environment variables (AWS_ACCESS_KEY_ID, AWS_SECRET_ACCESS_KEY,
#      AWS_REGION, S3_BUCKET, MASSIVE_API_KEY, ...)
#   2. config.json in the current directory (see config.example.json)
#   3. Built-in defaults (non-secret values only)
# On Kaggle: set secrets via notebook settings (Add-ons -> Secrets), which
# are injected as environment variables.
CONFIG_FILE = "config.json"


def get_secret(name, default=""):
    val = os.environ.get(name)
    if val:
        return val
    if Path(CONFIG_FILE).exists():
        try:
            with open(CONFIG_FILE) as f:
                data = json.load(f)
            if name in data:
                return str(data[name])
        except (OSError, ValueError):
            pass
    return default


class Config:
    def __init__(self):
        self.aws_access_key_id = ""
        self.aws_secret_access_key = ""
        self.aws_region = "us-east-1"
        self.s3_bucket = "market-data-zw"
        self.massive_api_key = ""

        # Paths (S3 keys under the bucket)
        self.types_prefix = "parquet_data/types"
        self.tickers_prefix = "parquet_data/summary/tickers"
        self.ticker_details_prefix = "parquet_data/summary/ticker_yahoo_details"
        self.minute_staging_prefix = "parquet_data/minute_data_staging"
        self.minute_final_prefix = "parquet_data/minute_data_final"
        self.minute_summary_prefix = "parquet_data/summary/minute_summary"
        self.daily_volume_prefix = "parquet_data/summary/daily_volume"
        self.correlation_prefix = "parquet_data/strategies/correlation"
        self.backtest_prefix = "parquet_data/backtest"
        self.backtest_metrics_prefix = "parquet_data/analysis/backtest_metrics"

        # Spark
        self.spark_executor_memory = "24g"
        self.spark_executor_cores = 4
        self.spark_driver_memory = "24g"
        self.spark_tmp = "/tmp/spark"


def load_config():
    cfg = Config()
    if Path(CONFIG_FILE).exists():
        try:
            with open(CONFIG_FILE) as f:
                data = json.load(f)
            for key, value in data.items():
                if hasattr(cfg, key):
                    setattr(cfg, key, value)
        except (OSError, ValueError) as e:
            print(f"[config] WARNING: could not load {CONFIG_FILE}: {e}")

    env_map = {
        "AWS_ACCESS_KEY_ID": "aws_access_key_id",
        "AWS_SECRET_ACCESS_KEY": "aws_secret_access_key",
        "AWS_REGION": "aws_region",
        "S3_BUCKET": "s3_bucket",
        "MASSIVE_API_KEY": "massive_api_key",
        "SPARK_DRIVER_MEMORY": "spark_driver_memory",
        "SPARK_EXECUTOR_MEMORY": "spark_executor_memory",
        "SPARK_EXECUTOR_CORES": "spark_executor_cores",
    }
    for env_name, attr in env_map.items():
        val = os.environ.get(env_name)
        if val:
            if attr == "spark_executor_cores":
                val = int(val)
            setattr(cfg, attr, val)
    return cfg
# --- S3 helpers -----------------------------------------------------------
def s3_client(cfg):
    from botocore.config import Config as BotocoreConfig
    config = BotocoreConfig(retries={"max_attempts": 5, "mode": "adaptive"},
                            connect_timeout=30, read_timeout=60)
    return boto3.client("s3",
                        aws_access_key_id=cfg.aws_access_key_id,
                        aws_secret_access_key=cfg.aws_secret_access_key,
                        region_name=cfg.aws_region,
                        config=config)


def upload_parquet(df, s3, bucket, key, compression="snappy"):
    buf = BytesIO()
    df.to_parquet(buf, index=False, engine="pyarrow", compression=compression,
                  coerce_timestamps="ms", allow_truncated_timestamps=True)
    buf.seek(0)
    s3.put_object(Bucket=bucket, Key=key, Body=buf.getvalue())


def download_parquet(s3, bucket, key):
    obj = s3.get_object(Bucket=bucket, Key=key)
    return pd.read_parquet(BytesIO(obj["Body"].read()))


def list_s3_keys(s3, bucket, prefix):
    keys = []
    paginator = s3.get_paginator("list_objects_v2")
    for page in paginator.paginate(Bucket=bucket, Prefix=prefix):
        for obj in page.get("Contents", []):
            keys.append(obj["Key"])
    return keys


def tickers_from_prefix(s3, bucket, prefix):
    """Ticker symbols from `<prefix>/<TICKER>.parquet` object keys."""
    return [k.split("/")[-1][:-len(".parquet")] for k in list_s3_keys(s3, bucket, prefix)
            if k.endswith(".parquet")]


def duckdb_s3_connect(cfg):
    con = duckdb.connect()
    con.execute("INSTALL httpfs; LOAD httpfs;")
    con.execute(f"SET s3_access_key_id='{cfg.aws_access_key_id}';")
    con.execute(f"SET s3_secret_access_key='{cfg.aws_secret_access_key}';")
    con.execute(f"SET s3_region='{cfg.aws_region}';")
    return con


def spark_session(cfg):
    from pyspark.sql import SparkSession
    spark = (
        SparkSession.builder
        .appName("MarketDataPlatform")
        .config("spark.jars.packages", "org.apache.hadoop:hadoop-aws:3.4.1")
        .config("spark.executor.memory", cfg.spark_executor_memory)
        .config("spark.executor.cores", str(cfg.spark_executor_cores))
        .config("spark.driver.memory", cfg.spark_driver_memory)
        .config("spark.hadoop.fs.s3a.access.key", cfg.aws_access_key_id)
        .config("spark.hadoop.fs.s3a.secret.key", cfg.aws_secret_access_key)
        .config("spark.hadoop.fs.s3a.endpoint", f"s3.{cfg.aws_region}.amazonaws.com")
        .config("spark.local.dir", cfg.spark_tmp)
        .config("spark.hadoop.tmp.dir", cfg.spark_tmp)
        .config("spark.sql.warehouse.dir", f"{cfg.spark_tmp}/warehouse")
        .getOrCreate()
    )
    spark.conf.set("spark.hadoop.fs.s3a.committer.name", "directory")
    spark.conf.set("spark.hadoop.mapreduce.fileoutputcommitter.algorithm.version", "2")
    spark.conf.set("spark.hadoop.fs.s3a.committer.staging.conflict-mode", "append")
    spark.conf.set("spark.sql.debug.maxToStringFields", "100")
    spark.conf.set("spark.sql.autoBroadcastJoinThreshold", "-1")
    spark.conf.set("spark.sql.ansi.enabled", "false")
    spark.conf.set("spark.sql.files.ignoreCorruptFiles", "true")
    spark.conf.set("spark.sql.parquet.mergeSchema", "true")
    spark.sparkContext.setLogLevel("ERROR")
    return spark

os.makedirs("/tmp/spark", exist_ok=True)

# --- Instantiate config + clients -----------------------------
cfg = load_config()
s3 = s3_client(cfg)
spark = spark_session(cfg)
print("Setup complete")
print(f"Bucket: {cfg.s3_bucket} | Region: {cfg.aws_region}")


In [ ]:
# ============================================================================
# Explicit schema -- prices/volume double, symbol/trades long. `volume` is
# double to absorb the int64/double drift across the per-ticker files.
# ============================================================================

from pyspark.sql import SparkSession, DataFrame, Window
from pyspark.sql import functions as F
from pyspark.sql.types import *

MINUTE_SCHEMA = StructType([
    StructField("symbol", StringType(), True),
    StructField("date",   LongType(),   True),   # epoch ms
    StructField("open",   DoubleType(), True),
    StructField("high",   DoubleType(), True),
    StructField("low",    DoubleType(), True),
    StructField("close",  DoubleType(), True),
    StructField("volume", DoubleType(), True),
    StructField("vwap",   DoubleType(), True),
    StructField("trades", LongType(),   True),
])

In [ ]:
# ============================================================================
# Load raw minute data + derive trading day + per-day ranks
# ============================================================================

spark = spark_session(cfg)
spark.conf.set("spark.sql.files.ignoreCorruptFiles", "true")

df_raw = spark.read.schema(MINUTE_SCHEMA).parquet(
    f"s3a://{cfg.s3_bucket}/{cfg.minute_final_prefix}/*.parquet"
)

df = (
    df_raw
    .withColumn("trade_date_time", F.to_timestamp(F.col("date") / 1000))
    .withColumn("trade_date", F.date_trunc("day", F.col("trade_date_time")))
    .drop("volume")
)

window_asc  = Window.partitionBy("symbol", "trade_date").orderBy("trade_date_time")
window_desc = Window.partitionBy("symbol", "trade_date").orderBy(F.col("trade_date_time").desc())

df = (
    df
    .withColumn("rn_asc",  F.row_number().over(window_asc))
    .withColumn("rn_desc", F.row_number().over(window_desc))
)
print("Schema:")
df.printSchema()

In [ ]:
# ============================================================================
# Daily volume & trades summary (all tickers, per trading day)
# ============================================================================

df_daily_vol = (
    df_raw
    .withColumn("trade_date", F.date_trunc("day", F.to_timestamp(F.col("date") / 1000)))
    .groupBy("symbol", "trade_date")
    .agg(
        F.sum("volume").alias("volume"),
        F.sum("trades").alias("trades"),
        F.last("close").alias("close"),
    )
)

df_daily_vol.write.mode("overwrite").parquet(
    f"s3a://{cfg.s3_bucket}/{cfg.daily_volume_prefix}/data.parquet"
)
print(f"Wrote {cfg.daily_volume_prefix}/data.parquet")

In [ ]:
# ============================================================================
# Minute summary: keep first + last minute bar of each (symbol, day)
# ============================================================================

df_filtered = df.filter((F.col("rn_asc") == 1) | (F.col("rn_desc") == 1))

df_filtered.write.mode("overwrite").parquet(
    f"s3a://{cfg.s3_bucket}/{cfg.minute_summary_prefix}/data.parquet"
)
print(f"Wrote {cfg.minute_summary_prefix}/data.parquet")

In [ ]:
# ============================================================================
# Optional: daily OHLCV aggregation (the README's canonical benchmark workload)
# ============================================================================

df_daily_ohlcv = (
    df_raw
    .withColumn("trade_date", F.date_trunc("day", F.to_timestamp(F.col("date") / 1000)))
    .groupBy("symbol", "trade_date")
    .agg(
        F.first("open", ignorenulls=True).alias("open"),
        F.max("high").alias("high"),
        F.min("low").alias("low"),
        F.last("close", ignorenulls=True).alias("close"),
        F.sum("volume").alias("volume"),
        F.sum("trades").alias("trades"),
        (F.sum(F.col("vwap") * F.col("volume"))
         / F.when(F.sum("volume") == 0, F.lit(1.0)).otherwise(F.sum("volume"))).alias("vwap"),
    )
)

df_daily_ohlcv.write.mode("overwrite").parquet(
    f"s3a://{cfg.s3_bucket}/parquet_data/summary/daily_ohlcv/data.parquet"
)
print("Wrote summary/daily_ohlcv/data.parquet")

spark.stop()